# Phase E — Safety Validation and Threshold Sensitivity

## Step 12: Realistic Harmful Recovery Experiments

### Research Question

Can the SHML safety controller identify harmful BILL_AMT1
recovery candidates when replacement values fall within
the observed training distribution?

### Experimental Controls

- Baseline model: Logistic Regression
- Dataset: UCI Default of Credit Card Clients
- Corrupted feature: BILL_AMT1
- Missingness level: 60%
- Evaluation: F1, recall, ROC-AUC
- Safety thresholds: 10% relative F1 and recall loss
- Random seed: 42

In [1]:

# Step 12A — Initialize SHML environment

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in (cwd, *cwd.parents)
        if (p / "src" / "run_pipeline.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the SHML repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "working"
    / "default of credit card clients.xls"
)

df = pd.read_excel(DATA_PATH, header=1, engine="xlrd")

target = "default payment next month"

X = df.drop(columns=["ID", target])
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_scaled, y_train)

healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

assert len(X_train) == 21000
assert len(X_val) == 4500
assert len(X_test) == 4500
assert abs(healthy_metrics["f1"] - 0.356983) < 0.0001

print("SHML initialized successfully.")
print("Healthy F1:", round(healthy_metrics["f1"], 6))
print("Healthy recall:", round(healthy_metrics["recall"], 6))
print("Healthy AUC:", round(healthy_metrics["roc_auc"], 6))


SHML initialized successfully.
Healthy F1: 0.356983
Healthy recall: 0.240201
Healthy AUC: 0.713025


In [2]:

# Step 12B — Define training-based replacement values

feature = "BILL_AMT1"

replacement_values = {
    "Training Minimum": float(X_train[feature].min()),
    "25th Percentile": float(X_train[feature].quantile(0.25)),
    "Training Median": float(X_train[feature].median()),
    "75th Percentile": float(X_train[feature].quantile(0.75)),
    "Training Maximum": float(X_train[feature].max())
}

replacement_df = pd.DataFrame([
    {
        "Strategy": name,
        "Replacement_Value": value
    }
    for name, value in replacement_values.items()
])

display(replacement_df)


,Strategy,Replacement_Value
0,Training Minimum,-165580.00
1,25th Percentile,3509.75
2,Training Median,22225.50
3,75th Percentile,67062.75
4,Training Maximum,964511.00


In [3]:

# Step 12C — Evaluate candidate repairs

seed = 42
rate = 0.60

X_corrupted = X_val.copy(deep=True)

rng = np.random.default_rng(seed)

n_missing = int(len(X_corrupted) * rate)

positions = rng.choice(
    len(X_corrupted),
    size=n_missing,
    replace=False
)

mask = np.zeros(len(X_corrupted), dtype=bool)
mask[positions] = True

X_corrupted.iloc[
    positions,
    X_corrupted.columns.get_loc(feature)
] = np.nan

sensitivity_results = []

for strategy, replacement in replacement_values.items():

    X_candidate = X_corrupted.copy(deep=True)

    X_candidate.loc[
        mask, feature
    ] = replacement

    # Verify that only injected positions changed
    assert not X_candidate.isna().any().any()

    pd.testing.assert_frame_equal(
        X_candidate.drop(columns=[feature]),
        X_val.drop(columns=[feature])
    )

    np.testing.assert_allclose(
        X_candidate.loc[~mask, feature].to_numpy(),
        X_val.loc[~mask, feature].to_numpy(),
        rtol=0,
        atol=0
    )

    metrics = evaluate_model(
        X_batch=X_candidate,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    safety = evaluate_recovery_safety(
        recovered_metrics=metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=True,
        max_f1_loss=0.10,
        max_recall_loss=0.10
    )

    sensitivity_results.append({
        "Strategy": strategy,
        "Replacement_Value": replacement,
        "Missingness": rate,
        "F1": metrics["f1"],
        "Recall": metrics["recall"],
        "ROC_AUC": metrics["roc_auc"],
        "F1_Loss_Pct": safety["f1_loss_pct"],
        "Recall_Loss_Pct": safety["recall_loss_pct"],
        "Decision": safety["decision"],
        "Reason": safety["reason"]
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

display(sensitivity_df.round(6))


,Strategy,Replacement_Value,Missingness,F1,Recall,ROC_AUC,F1_Loss_Pct,Recall_Loss_Pct,Decision,Reason
0,Training Minimum,-165580.00,0.6,0.444855,0.362814,0.685080,-24.615297,-51.046025,ACCEPT,All checks passed
1,25th Percentile,3509.75,0.6,0.392883,0.277387,0.705153,-10.056433,-15.481172,ACCEPT,All checks passed
2,Training Median,22225.50,0.6,0.384949,0.267337,0.706199,-7.834138,-11.297071,ACCEPT,All checks passed
3,75th Percentile,67062.75,0.6,0.353208,0.235176,0.706885,1.057551,2.092050,ACCEPT,All checks passed
4,Training Maximum,964511.00,0.6,0.153153,0.085427,0.614443,57.097893,64.435146,REJECT,F1 loss exceeds tolerance; Recall loss exceeds...



### Observation E.3 — Distribution-Grounded Safety Testing

The SHML safety controller was evaluated using five
replacement values derived from the BILL_AMT1 training
distribution.

Controlled missingness was introduced into 60% of
validation observations using random seed 42.

#### Main Findings

Four replacement candidates were accepted, while
the training maximum candidate was rejected.

The training maximum (964,511) reduced F1 to 0.153153
and recall to 0.085427.

This represented relative losses of 57.10% in F1
and 64.44% in recall.

Both exceeded the existing 10% safety tolerances.

The training minimum (-165,580) increased F1 to
0.444855 but reduced ROC-AUC to 0.685080.

This candidate was accepted because the safety
controller currently evaluates F1, recall, metric
validity, and structural integrity rather than
enforcing a separate ROC-AUC loss threshold.

#### Interpretation

These results demonstrate that the safety controller
can reject a severely harmful recovery candidate using
a replacement value observed in the training dataset.

However, a value being observed in training does not
make its use as a constant imputation realistic.

The findings also reveal that safety acceptance is
dependent on the selected performance constraints.

#### Limitations

The experiment used one missingness severity and
one random seed.

The replacement strategies are controlled stress
tests rather than independently validated imputation
algorithms.

The results do not establish rejection reliability
across different corruption patterns or datasets.

#### Next Step

Repeat the sensitivity experiment across multiple
missingness levels and random seeds while retaining
the original safety thresholds.


In [4]:

# Step 12D — Repeated safety sensitivity experiments

repeated_safety_results = []

missingness_levels = [0.05, 0.15, 0.30, 0.60]
seeds = range(20)

for seed in seeds:
    for rate in missingness_levels:

        X_corrupted = X_val.copy(deep=True)

        rng = np.random.default_rng(seed)

        n_missing = int(len(X_corrupted) * rate)

        positions = rng.choice(
            len(X_corrupted),
            size=n_missing,
            replace=False
        )

        mask = np.zeros(len(X_corrupted), dtype=bool)
        mask[positions] = True

        X_corrupted.iloc[
            positions,
            X_corrupted.columns.get_loc("BILL_AMT1")
        ] = np.nan

        for strategy, replacement in replacement_values.items():

            X_candidate = X_corrupted.copy(deep=True)

            X_candidate.loc[
                mask, "BILL_AMT1"
            ] = replacement

            # Verify targeted reconstruction
            assert not X_candidate.isna().any().any()

            pd.testing.assert_frame_equal(
                X_candidate.drop(columns=["BILL_AMT1"]),
                X_val.drop(columns=["BILL_AMT1"])
            )

            np.testing.assert_allclose(
                X_candidate.loc[~mask, "BILL_AMT1"].to_numpy(),
                X_val.loc[~mask, "BILL_AMT1"].to_numpy(),
                rtol=0,
                atol=0
            )

            metrics = evaluate_model(
                X_batch=X_candidate,
                y_true=y_val,
                model=baseline_model,
                scaler=scaler
            )

            safety = evaluate_recovery_safety(
                recovered_metrics=metrics,
                healthy_metrics=healthy_metrics,
                data_integrity_passed=True,
                max_f1_loss=0.10,
                max_recall_loss=0.10
            )

            repeated_safety_results.append({
                "Seed": seed,
                "Missingness": rate,
                "Strategy": strategy,
                "F1": metrics["f1"],
                "Recall": metrics["recall"],
                "ROC_AUC": metrics["roc_auc"],
                "F1_Loss_Pct": safety["f1_loss_pct"],
                "Recall_Loss_Pct": safety["recall_loss_pct"],
                "Decision": safety["decision"]
            })

repeated_safety_df = pd.DataFrame(repeated_safety_results)

assert len(repeated_safety_df) == 400

rejection_summary = (
    repeated_safety_df
    .groupby(["Missingness", "Strategy"])
    .agg(
        Evaluations=("Decision", "size"),
        Accepted=(
            "Decision",
            lambda s: (s == "ACCEPT").sum()
        ),
        Rejected=(
            "Decision",
            lambda s: (s == "REJECT").sum()
        ),
        Mean_F1=("F1", "mean"),
        Mean_AUC=("ROC_AUC", "mean")
    )
    .reset_index()
)

rejection_summary["Rejection_Rate_Pct"] = (
    rejection_summary["Rejected"]
    / rejection_summary["Evaluations"]
    * 100
)

display(rejection_summary.round(4))


,Missingness,Strategy,Evaluations,Accepted,Rejected,Mean_F1,Mean_AUC,Rejection_Rate_Pct
0,0.05,25th Percentile,20,20,0,0.3603,0.7118,0.0
1,0.05,75th Percentile,20,20,0,0.3568,0.7122,0.0
2,0.05,Training Maximum,20,20,0,0.3447,0.6960,0.0
3,0.05,Training Median,20,20,0,0.3593,0.7120,0.0
4,0.05,Training Minimum,20,20,0,0.3651,0.7086,0.0
5,0.15,25th Percentile,20,20,0,0.3673,0.7108,0.0
6,0.15,75th Percentile,20,20,0,0.3569,0.7111,0.0
7,0.15,Training Maximum,20,0,20,0.3165,0.6593,100.0
8,0.15,Training Median,20,20,0,0.3648,0.7112,0.0
9,0.15,Training Minimum,20,20,0,0.3858,0.7026,0.0



### Observation E.4 — Repeated Safety Sensitivity Analysis

The SHML safety controller was evaluated using five
distribution-grounded BILL_AMT1 replacement strategies,
four missingness levels, and 20 random injection seeds.

A total of 400 candidate evaluations were performed.

#### Results

- Total evaluations: 400
- Accepted: 340 (85%)
- Rejected: 60 (15%)

The training-maximum replacement was rejected in all
20 seeds at 15%, 30%, and 60% missingness.

At 5% missingness, all 20 training-maximum candidates
were accepted.

The remaining four replacement strategies were
accepted across all tested seeds and severity levels.

#### Interpretation

The existing safety controller demonstrates consistent
rejection of a severely damaging replacement strategy
at higher missingness levels.

The first tested severity with consistent rejection
was 15%. The exact transition boundary has not been
established.

The results also reveal a metric-dependent limitation.

Some accepted candidates increased F1 while decreasing
ROC-AUC relative to the healthy reference.

This occurs because the current safety gate constrains
F1 and recall losses but does not impose a separate
ROC-AUC loss limit.

#### Scientific Limitations

The experiment used one trained model and one
validation dataset.

The 20 seeds varied injected missingness patterns,
not the model-training procedure.

The replacement values came from the training
distribution, but constant imputation with an extreme
value is not necessarily realistic.

Therefore, the results demonstrate repeatability under
controlled conditions rather than universal safety.

#### Conclusion

H3 is supported under the tested experimental
conditions for both acceptance and rejection behavior.

Further work should investigate the sensitivity of
safety decisions to the configured evaluation criteria
and loss tolerances.


## Step 13 — Safety Threshold Sensitivity Analysis

### Objective
Evaluate how different safety tolerances influence candidate
acceptance and rejection decisions while keeping the original
model, missingness scenarios, and recovery candidates fixed.

In [5]:

# Step 13A — F1 and Recall Threshold Sensitivity

import pandas as pd
import numpy as np

# Existing Step 12D results
assert len(repeated_safety_df) == 400

# Alternative thresholds for sensitivity analysis only
tolerances = [0.05, 0.10, 0.15, 0.20]

threshold_results = []

for tolerance in tolerances:

    # Positive loss means degradation
    # Negative loss means improvement
    rejected = (
        (repeated_safety_df["F1_Loss_Pct"] > tolerance * 100)
        |
        (repeated_safety_df["Recall_Loss_Pct"] > tolerance * 100)
    )

    threshold_results.append({
        "Tolerance_Pct": tolerance * 100,
        "Total": len(repeated_safety_df),
        "Accepted": int((~rejected).sum()),
        "Rejected": int(rejected.sum()),
        "Rejection_Rate_Pct": rejected.mean() * 100
    })

threshold_df = pd.DataFrame(threshold_results)

# Verify the original 10% rule reproduces Step 12D
original = threshold_df[
    threshold_df["Tolerance_Pct"] == 10
].iloc[0]

assert original["Accepted"] == 340
assert original["Rejected"] == 60

display(threshold_df.round(2))


,Tolerance_Pct,Total,Accepted,Rejected,Rejection_Rate_Pct
0,5.0,400,329,71,17.75
1,10.0,400,340,60,15.00
2,15.0,400,349,51,12.75
3,20.0,400,360,40,10.00


In [6]:

# Step 13B — Explore adding ROC-AUC safety checks

healthy_auc = healthy_metrics["roc_auc"]

auc_results = repeated_safety_df.copy()

auc_results["AUC_Loss_Pct"] = (
    (
        healthy_auc - auc_results["ROC_AUC"]
    )
    / healthy_auc
    * 100
)

# Existing controller decisions
auc_results["Original_Decision"] = auc_results["Decision"]

# Illustrative additional ROC-AUC tolerance
auc_tolerance_pct = 2.0

auc_results["Proposed_Decision"] = np.where(
    (auc_results["Original_Decision"] == "REJECT")
    |
    (auc_results["AUC_Loss_Pct"] > auc_tolerance_pct),
    "REJECT",
    "ACCEPT"
)

# Compare old and proposed decisions
comparison = pd.crosstab(
    auc_results["Original_Decision"],
    auc_results["Proposed_Decision"]
)

display(comparison)

# Identify candidates that the additional rule would reject
new_rejections = auc_results[
    (auc_results["Original_Decision"] == "ACCEPT")
    &
    (auc_results["Proposed_Decision"] == "REJECT")
]

print(
    "Additional rejections under illustrative AUC rule:",
    len(new_rejections)
)

display(
    new_rejections.groupby(
        ["Missingness", "Strategy"]
    ).size().reset_index(
        name="Additional_Rejections"
    )
)


Proposed_Decision,ACCEPT,REJECT
Original_Decision,,
ACCEPT,282,58
REJECT,0,60


Additional rejections under illustrative AUC rule: 58


,Missingness,Strategy,Additional_Rejections
0,0.05,Training Maximum,15
1,0.15,Training Minimum,4
2,0.30,Training Minimum,17
3,0.60,25th Percentile,1
4,0.60,Training Median,1
5,0.60,Training Minimum,20


In [7]:
import plotly.graph_objects as go

fig = go.Figure(go.Scatter(
    x=threshold_df["Tolerance_Pct"], y=threshold_df["Rejected"], mode="lines+markers",
    hovertemplate="Tolerance: %{x:g}%<br>Rejected candidates: %{y}<extra></extra>",
))
fig.add_vline(x=10, line_dash="dash", line_color="gray",
              annotation_text="Existing tolerance (10%)", annotation_position="top right")
fig.update_layout(
    title="SHML Safety Threshold Sensitivity",
    xaxis=dict(title="Maximum F1 and recall loss (%)", tickvals=threshold_df["Tolerance_Pct"]),
    yaxis_title="Number of rejected candidates",
)

FIGURES_DIR = PROJECT_ROOT / "results" / "research" / "bill_amt1" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.write_html(FIGURES_DIR / "safety_threshold_sensitivity.html", include_plotlyjs="cdn")
fig.show(renderer="plotly_mimetype")



### Observation E.5 — Safety Threshold Sensitivity

The safety controller was evaluated using alternative
F1 and recall loss tolerances while keeping the trained
model, validation dataset, and 400 repair candidates fixed.

#### Results

| Tolerance | Accepted | Rejected |
|---|---:|---:|
| 5% | 329 | 71 |
| 10% | 340 | 60 |
| 15% | 349 | 51 |
| 20% | 360 | 40 |

The original 10% tolerance rejected 60 candidates.

Stricter tolerances increased rejections, whereas
more permissive tolerances reduced them.

#### Additional ROC-AUC Constraint

An exploratory 2% relative ROC-AUC loss constraint
resulted in 58 additional rejections.

The proposed combined rule rejected 118 of 400 candidates.

These findings demonstrate that SHML safety decisions
depend on the performance metrics and allowable losses.

#### Interpretation

The current controller successfully identifies substantial
F1 and recall degradation under the tested conditions.

However, acceptance does not guarantee preservation
of ROC-AUC or other unmonitored metrics.

The 2% ROC-AUC threshold is exploratory and has not
been established as optimal.

The original 10% F1 and recall safety tolerances remain
unchanged for the primary experiment.


# Phase F — Recovery Benefit Evaluation

## Step 14: Measuring the Benefit of Healing

### Research Question

Does SHML improve predictive performance relative to
a predefined fallback policy under BILL_AMT1 missingness?

### Comparison Groups

1. Healthy reference — original validation observations
   without injected missingness.

2. Unavailable / no-scoring baseline — corrupted
   observations cannot be passed directly to the
   existing Logistic Regression pipeline.

3. Fallback policy — use a predefined, training-derived
   constant to obtain valid predictions.

4. SHML recovery — evaluate zero, median, and
   regression-based recovery candidates.

### Primary Metric
F1-score at the predefined classification threshold.

### Secondary Metrics
Recall and ROC-AUC.

### Scientific Limitation
Fallback imputation is itself a corrective action.
Therefore, comparisons against a fallback policy
measure the incremental benefit of the recovery strategy,
not improvement over literal non-intervention.